# SciCode-Bench: The "Visual Cortex" (Topological Inference)

## **Objective**
This benchmark evaluates an LLM's ability to simulate a **Visual Cortex** using pure Python logic. Inspired by the **ARC-AGI** (Abstraction and Reasoning Corpus), it tests whether a model can perceive a noisy 2D matrix and perform **Topological Inference**—recognizing a "concept" (like a Ring) even when the physical data is disoriented or has significant gaps.

## **The Task: "The Broken Enclosure"**
The model must analyze a 30x30 grid and identify a specific "Target" while ignoring "Distractors" that share similar statistical properties (like pixel density) but different topological structures.

**Subtasks:**
1.  **Raw Perception:** Perform a statistical scan of the grid to identify active color channels and pixel counts.
2.  **Visual Description:** Extract geometric metadata (Bounding Boxes, Density, and Centroids) to abstract raw pixels into "objects."
3.  **Recognition & Segmentation:** Classify objects as 'Ring' (Hollow), 'Block' (Solid), or 'Noise' and return the **exact pixel coordinates** for the Target Broken Enclosure.

## **Requirements**
* **Environment:** Google Colab Pro.
* **Agent Evaluation:** Present the Subtask prompts to the Gemini 3 Pro agent.
* **Verification:** All subtasks produce machine-verifiable outputs (JSON or Coordinate Lists) verified by deterministic unit tests.
* **Constraint:** No external Computer Vision libraries (e.g., OpenCV) are permitted.

In [ ]:
import numpy as np
import json
from scipy.ndimage import label, center_of_mass

def get_benchmark_grid():
    """
    Generates a 30x30 grid with complex topological objects.
    Color 1: Broken Enclosure (Target) - 29 pixels
    Color 2: Pseudo-Ring C-Shape (Distractor) - 26 pixels
    Color 3: Sparse Spiral (Distractor)
    Color 4: Noise Cluster
    """
    grid = np.zeros((30, 30), dtype=int)

    # --- Object 1: The Target "Broken Enclosure" (Color 1) ---
    grid[2, 2:12] = 1
    grid[11, 2:12] = 1
    grid[3:6, 2] = 1
    grid[8:11, 2] = 1
    grid[3:5, 11] = 1
    grid[10:11, 11] = 1

    # --- Object 2: Distractor "The Pseudo-Ring" (Color 2) ---
    grid[2:12, 18] = 2
    grid[2, 19:27] = 2
    grid[11, 19:27] = 2

    # --- Object 3: Distractor "The Sparse Spiral" (Color 3) ---
    grid[18:27, 12] = 3; grid[18, 12:21] = 3; grid[18:25, 21] = 3
    grid[25, 14:22] = 3; grid[20:25, 14] = 3; grid[20, 14:19] = 3

    # --- Object 4: Noise (Color 4) ---
    for r, c in [(20, 25), (21, 26), (22, 24)]:
        grid[r, c] = 4

    return grid

print("✅ Benchmark Environment and 30x30 Grid Ready.")

In [ ]:
# --- Subtask 1: Raw Perception (Counts) ---
# Prompt: "Scan the 30x30 grid and report the raw pixel counts for each color channel."

def get_color_counts(grid):
    """
    TODO: Implementation required.
    Should return a dictionary of {color_id: pixel_count}.
    """
    # [Agent: Implement logic here]
    return {}

def test_subtask_1():
    print("Testing Subtask 1 (Raw Perception)...", end="")
    grid = get_benchmark_grid()
    counts = get_color_counts(grid)
    assert counts[1] == 29, f"Expected 29 pixels for Color 1, got {counts.get(1)}"
    assert counts[2] == 26, f"Expected 26 pixels for Color 2, got {counts.get(2)}"
    print(" PASSED ✅")

In [ ]:
# --- Subtask 2: Visual Description (Geometry Extraction) ---
# Prompt: "Analyze the objects in the grid. For each color, calculate
# Bounding Box (Height/Width), Density, and provide the exact list of pixels."

def extract_metadata(grid):
    """
    TODO: Implementation required.
    Should return a dictionary where keys are color_ids and values are dicts
    containing 'bbox_hw', 'density', and 'pixels'.
    """
    # [Agent: Implement logic here]
    return {}

def test_subtask_2():
    print("Testing Subtask 2 (Description)...", end="")
    grid = get_benchmark_grid()
    meta = extract_metadata(grid)
    assert meta[1]["bbox_hw"] == (10, 10)
    assert 0.28 < meta[1]["density"] < 0.30
    print(" PASSED ✅")

In [ ]:
# --- Subtask 3: Recognition & Segmentation (The Headroom Task) ---
# Prompt: "Identify the 'Broken Enclosure' object (Target).
# It is defined by its hollow center void. Return its exact pixel coordinates.
# Distinguish it from sparse blocks or open 'C' shapes using topological reasoning."

def identify_target_enclosure(grid):
    """
    TODO: Implementation required.
    Identify the specific Color_ID associated with the Broken Enclosure
    and return its sorted list of (row, col) coordinates.
    """
    # [Agent: Implement logic here]
    return []

def test_subtask_3():
    print("Testing Subtask 3 (Segmentation)...", end="")
    grid = get_benchmark_grid()
    res = identify_target_enclosure(grid)
    assert len(res) == 29, f"Expected 29 pixels, found {len(res)}"
    assert (6, 2) not in res, "Error: Identified gap pixel as object pixel."
    print(" PASSED ✅")

In [ ]:
# --- Final Benchmark Runner ---
def run_full_benchmark():
    import json
    results = {"subtasks": {}, "overall_status": "FAIL"}

    test_suite = [
        ("Subtask_1_Perception", test_subtask_1),
        ("Subtask_2_Description", test_subtask_2),
        ("Subtask_3_Segmentation", test_subtask_3)
    ]

    for name, test_func in test_suite:
        try:
            test_func()
            results["subtasks"][name] = "PASS"
        except Exception as e:
            results["subtasks"][name] = f"FAIL: {e}"

    if all(v == "PASS" for v in results["subtasks"].values()):
        results["overall_status"] = "SUCCESS"

    print("\n--- Final Machine-Verifiable Output ---")
    print(json.dumps(results, indent=2))

if __name__ == "__main__":
    run_full_benchmark()